# 📊 Camada Gold — Validação e Exploração

Este notebook valida os dados gerados pelo pipeline `src/gold_pipeline.py` e exibe distribuições relevantes para o sistema de recomendação.

**Tabelas Gold disponíveis:**
| Tabela | Caso de Uso |
|---|---|
| `gold_user_product_interactions` | Collaborative filtering |
| `gold_product_features` | Content-based filtering |
| `gold_customer_profile` | Perfil / segmentação de clientes |
| `gold_category_affinity` | Recomendação cross-category |
| `gold_product_cooccurrence` | "Também comprou" / regras de associação |
| `gold_top_products_by_segment` | Cold start / fallback |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import os

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

GOLD_DIR = os.path.join('..', 'data', 'gold')

def load(name):
    return pd.read_csv(os.path.join(GOLD_DIR, f'{name}.csv'))

interactions = load('gold_user_product_interactions')
products     = load('gold_product_features')
profile      = load('gold_customer_profile')
affinity     = load('gold_category_affinity')
cooccurrence = load('gold_product_cooccurrence')
top_segments = load('gold_top_products_by_segment')

print('✓ Todas as tabelas carregadas')

## 1. Visão Geral — Contagem de Linhas e Nulos

In [ ]:
tables = {
    'user_product_interactions': interactions,
    'product_features':          products,
    'customer_profile':          profile,
    'category_affinity':         affinity,
    'product_cooccurrence':      cooccurrence,
    'top_products_by_segment':   top_segments,
}

summary = []
for name, df in tables.items():
    nulls = df.isnull().sum().sum()
    summary.append({'Tabela': name, 'Linhas': len(df), 'Colunas': len(df.columns), 'Nulos totais': nulls})

pd.DataFrame(summary).set_index('Tabela').style.format({'Linhas': '{:,}', 'Nulos totais': '{:,}'})

## 2. Interações Usuário × Produto
### 2.1 Distribuição do Implicit Score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('gold_user_product_interactions', fontweight='bold')

# Implicit score
interactions['implicit_score'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição do Implicit Score')
axes[0].set_xlabel('implicit_score')

# Review score
interactions['review_score'].dropna().hist(bins=5, ax=axes[1], color='coral', edgecolor='white', rwidth=0.8)
axes[1].set_title('Distribuição do Review Score')
axes[1].set_xlabel('review_score (1-5)')

# Sentiment
sent_counts = interactions['sentiment'].value_counts()
axes[2].bar(sent_counts.index, sent_counts.values, color=['#2ecc71', '#e74c3c', '#95a5a6'])
axes[2].set_title('Distribuição de Sentimento')

plt.tight_layout()
plt.show()

print(f"\nEstatísticas do implicit_score:")
print(interactions['implicit_score'].describe().round(3))
print(f"\nClientes únicos: {interactions['customer_unique_id'].nunique():,}")
print(f"Produtos únicos: {interactions['product_id'].nunique():,}")
print(f"Reviews presentes: {interactions['review_score'].notna().sum():,} ({interactions['review_score'].notna().mean()*100:.1f}%)")

### 2.2 Top 10 Categorias nas Interações

In [ ]:
top_cats = interactions['product_category_name_english'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 4))
top_cats.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Categorias nas Interações')
ax.set_xlabel('Nº de Interações')
plt.tight_layout()
plt.show()

## 3. Features de Produto
### 3.1 Distribuição de Preço Médio e Avaliação

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('gold_product_features', fontweight='bold')

# Preço médio (sem outliers)
price_clip = products['avg_price'].clip(upper=products['avg_price'].quantile(0.95))
price_clip.hist(bins=40, ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].set_title('Preço Médio (p95 clipped)')
axes[0].set_xlabel('R$')

# Avaliação média
products['avg_review_score'].dropna().hist(bins=20, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Avg Review Score por Produto')

# Positive sentiment ratio
products['positive_sentiment_ratio'].dropna().hist(bins=20, ax=axes[2], color='#2ecc71', edgecolor='white')
axes[2].set_title('% Reviews Positivos')

plt.tight_layout()
plt.show()

print(f"Produtos únicos: {len(products):,}")
print(f"Categorias únicas: {products['product_category_name_english'].nunique()}")
print(f"\nPreço médio geral: R$ {products['avg_price'].mean():.2f}")
print(f"Avaliação média geral: {products['avg_review_score'].mean():.2f}")

### 3.2 Top 10 Produtos Mais Vendidos

In [ ]:
top10 = products.nlargest(10, 'total_orders')[['product_id', 'product_category_name_english', 'total_orders', 'avg_review_score', 'avg_price']]
top10['product_id_short'] = top10['product_id'].str[:8] + '...'

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(top10['product_id_short'], top10['total_orders'], color='steelblue')
for bar, cat in zip(bars, top10['product_category_name_english']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f' [{cat}]', va='center', fontsize=8, color='gray')
ax.set_title('Top 10 Produtos por Total de Pedidos')
ax.set_xlabel('Total de Pedidos')
plt.tight_layout()
plt.show()

top10[['product_category_name_english', 'total_orders', 'avg_review_score', 'avg_price']].head(10)

## 4. Perfil de Cliente
### 4.1 Distribuição Geográfica

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('gold_customer_profile — Localidade', fontweight='bold')

# Por estado
state_counts = profile['customer_state'].value_counts().head(15)
state_counts.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Clientes por Estado (top 15)')
axes[0].set_xlabel('Nº de Clientes')

# Categoria preferida
pref_cat = profile['preferred_category'].value_counts().head(10)
pref_cat.sort_values().plot(kind='barh', ax=axes[1], color='mediumpurple')
axes[1].set_title('Categoria Preferida (top 10)')

plt.tight_layout()
plt.show()

print(f"Clientes únicos: {len(profile):,}")
print(f"Estados cobertos: {profile['customer_state'].nunique()}")
print(f"Gasto médio por cliente: R$ {profile['total_spent'].mean():.2f}")

### 4.2 Métodos de Pagamento e Parcelamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pagamento preferido
pay = profile['preferred_payment_type'].value_counts()
axes[0].pie(pay.values, labels=pay.index, autopct='%1.1f%%', startangle=90,
            colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
axes[0].set_title('Método de Pagamento Preferido')

# Média de parcelas
profile['avg_installments'].dropna().hist(bins=20, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribuição de Parcelas Médias')
axes[1].set_xlabel('Nº médio de parcelas')

plt.tight_layout()
plt.show()

## 5. Afinidade por Categoria
### 5.1 Distribuição do Affinity Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('gold_category_affinity', fontweight='bold')

# Distribuição geral
affinity['affinity_score'].hist(bins=50, ax=axes[0], color='#e67e22', edgecolor='white')
axes[0].set_title('Distribuição do Affinity Score')
axes[0].set_xlabel('affinity_score (0-1)')

# Top categorias por afinidade média
cat_aff = affinity.groupby('product_category_name_english')['affinity_score'].mean().nlargest(10)
cat_aff.sort_values().plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Categorias com Maior Afinidade Média')

plt.tight_layout()
plt.show()

print(f"Pares cliente×categoria: {len(affinity):,}")
print(f"Categorias únicas: {affinity['product_category_name_english'].nunique()}")
print(f"\nAffinity score — range: [{affinity['affinity_score'].min():.3f}, {affinity['affinity_score'].max():.3f}]")

## 6. Co-ocorrência de Produtos
### 6.1 Pares com Maior Co-ocorrência (Jaccard)

In [ ]:
top_cooc = cooccurrence.nlargest(10, 'cooccurrence_score')[[
    'product_id_a', 'product_id_b', 'category_a', 'category_b',
    'cooccurrence_count', 'cooccurrence_score'
]].reset_index(drop=True)

top_cooc['product_id_a'] = top_cooc['product_id_a'].str[:8] + '...'
top_cooc['product_id_b'] = top_cooc['product_id_b'].str[:8] + '...'

print(f"Total de pares: {len(cooccurrence):,}")
print(f"\nTop 10 pares por Jaccard Score:")
display(top_cooc)

# Distribuição do co-occurrence score
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cooccurrence['cooccurrence_count'].clip(upper=20).hist(bins=20, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Distribuição de Co-ocorrências (clipped @20)')
axes[0].set_xlabel('Nº de pedidos com o par')

cooccurrence['cooccurrence_score'].hist(bins=30, ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Distribuição do Jaccard Score')
axes[1].set_xlabel('Jaccard similarity')

plt.tight_layout()
plt.show()

## 7. Cold Start — Top Produtos por Segmento

In [ ]:
print("Segmentos disponíveis:")
print(top_segments.groupby('segment_type')['segment_value'].nunique().to_string())

print("\n--- Top 10 Global ---")
display(top_segments[top_segments['segment_type'] == 'global'][[
    'rank', 'product_category_name_english', 'avg_review_score', 'avg_price', 'total_orders'
]].set_index('rank'))

print("\n--- Top 5 em SP ---")
display(top_segments[
    (top_segments['segment_type'] == 'state') & (top_segments['segment_value'] == 'SP')
][['rank', 'product_category_name_english', 'avg_review_score', 'avg_price']].head(5).set_index('rank'))

## 8. Resumo de Validação ✅

In [ ]:
checks = []

# Implicit score entre 0 e 1.5
score_ok = interactions['implicit_score'].between(0, 1.5).all()
checks.append(('implicit_score em [0, 1.5]', '✅' if score_ok else '❌'))

# Sem nulos em customer_unique_id
nulls_user = interactions['customer_unique_id'].isna().sum()
checks.append(('Sem nulos em customer_unique_id (interactions)', '✅' if nulls_user == 0 else f'❌ {nulls_user} nulos'))

# Sem nulos em product_id
nulls_prod = interactions['product_id'].isna().sum()
checks.append(('Sem nulos em product_id (interactions)', '✅' if nulls_prod == 0 else f'❌ {nulls_prod} nulos'))

# Affinity score entre 0 e 1
aff_ok = affinity['affinity_score'].between(0, 1).all()
checks.append(('affinity_score em [0, 1]', '✅' if aff_ok else '❌'))

# Top segmentos cobre global
global_ok = (top_segments['segment_type'] == 'global').any()
checks.append(('Segmento global presente', '✅' if global_ok else '❌'))

# 26 estados cobertos
n_states = top_segments[top_segments['segment_type'] == 'state']['segment_value'].nunique()
checks.append((f'Estados cobertos no cold start ({n_states}/27)', '✅' if n_states >= 25 else f'⚠️ {n_states}'))

# Cooccurrence score entre 0 e 1
cooc_ok = cooccurrence['cooccurrence_score'].between(0, 1).all()
checks.append(('cooccurrence_score (Jaccard) em [0, 1]', '✅' if cooc_ok else '❌'))

print("=" * 55)
print("  VALIDAÇÃO DA CAMADA GOLD")
print("=" * 55)
for label, status in checks:
    print(f"  {status}  {label}")
print("=" * 55)